# CREM: Porcentaje de Población de 65 o más
**Propósito**: Extrae la información sobre la evolución del porcentaje de población de 65 años o más según municipios, distritos y secciones censales a partir de la página oficial del CREM (Centro Regional de Estadística de Murcia), procesa y limpia los datos directamente en memoria y los almacena en una tabla Delta Lake en Spark.

In [1]:
import sys
import re
import requests
from bs4 import BeautifulSoup
import pandas as pd
from pyspark.sql import functions as F

sys.path.insert(0, '/tfm/python_notebooks')
from tfm_lib import utils as tfm_utils
from tfm_lib import crem_scrap as crem

In [2]:
# Definir la url de la web
url_base = "https://econet.carm.es/web/crem/inicio/-/crem/sicrem/PM2089//sec29.html"
table_name = "raw.crem.porcentaje_de_poblacion_de_65_o_mas"
column_suffix = "Porcentaje_de_poblacion_de_65_o_mas"

## Descarga y Parseo del Contenido HTML directamente en Memoria
Se realiza una petición HTTP para descargar el contenido HTML de la página oficial del CREM en memoria y procesarlo dinámicamente sin persistir archivos intermedios en el disco local.

In [3]:

# Hacer la solicitud
html = crem.get_html(url_base)

# Convertir HTML a BeautifulSoup object.
soup = BeautifulSoup(html, "html.parser")

table = soup.find("table", id="interiorTablaDatos")

if not table:
    raise Exception("No se encontró la tabla con id 'interiorTablaDatos' en el HTML")

# Obtener cabeceras (años)
thead = table.find("thead", id="cabeceraTablaDatos")
anos_columnas = []
for th in thead.find_all("th"):
    val = th.get_text(strip=True)
    if val:
        anos_columnas.append(val)

print(f"Años detectados: {anos_columnas}")

Años detectados: ['2023', '2022', '2021', '2020', '2019', '2018', '2017', '2016', '2015']


In [4]:
# Parsear filas de datos jerárquicos
tbody = table.find("tbody")
datos_filas = []

mapping_niveles = {
    1: "Región",
    2: "Municipio",
    3: "Distrito",
    4: "Sección"
}

for tr in tbody.find_all("tr"):
    th = tr.find("th", id="r")
    if not th:
        continue
    
    ul = th.find("ul")
    level_num = None
    if ul and ul.has_attr("class"):
        for c in ul["class"]:
            if c.startswith("level"):
                try:
                    level_num = int(c.replace("level", ""))
                except ValueError:
                    pass
                break
    
    label_raw = th.get_text(strip=True)
    label_clean = crem.clean_and_decode(label_raw)
    
    # Extraer código y nombre (ej. 'Abanilla - 30001' o 'Abanilla distrito 01 - 3000101')
    match = re.match(r"^(.*?)\s*-\s*(\d+)$", label_clean)
    if match:
        nombre = match.group(1).strip()
    else:
        nombre = label_clean

    if ',' in nombre: nombre=nombre.split(',')[1]+' '+nombre.split(',')[0]
    
    nivel = mapping_niveles.get(level_num, "Desconocido")
    
    tds = tr.find_all("td", id="d", class_="tbdato")
    values = []
    for td in tds:
        val_str = td.get_text(strip=True).replace(",", ".")
        try:
            val_float = float(val_str) if val_str not in ("", "-", "..") else None
        except ValueError:
            val_float = None
        values.append(val_float)
    
    if len(values) == len(anos_columnas):
        registro = {
            "nombre": crem.normalize_name(nombre),
            "nivel": nivel
        }
        for a, val in zip(anos_columnas, values):
            registro[a] = val
        datos_filas.append(registro)

print(f"Filas parseadas con éxito: {len(datos_filas)}")

Filas parseadas con éxito: 1350


## Integración con PySpark y Delta Lake
Se crea un Spark Session, se inicializa el Spark DataFrame a partir del listado en memoria, se normaliza su esquema con las utilidades de TFM y se guarda como Delta Table en la ruta final.

In [5]:
# Inicializar sesión de Spark usando las utilidades de TFM
spark = tfm_utils.get_spark_session(app_name="CREM_Porcentaje_Poblacion_65_o_mas")

# Crear el Spark DataFrame a partir de la lista de registros estructurados
df = (spark.createDataFrame(datos_filas)
          .filter(F.col("nivel").like("Municipio"))
          .drop("nivel")
     )

for anio in anos_columnas:
    df = df.withColumnRenamed(f"{anio}", f"{anio}_{column_suffix}")

# Normalizar las columnas del DataFrame de acuerdo al estilo del TFM (minúsculas, sin acentos, ordenadas)
df_normalized = tfm_utils.normalize_df(df)

# Mostrar los primeros registros de forma estructurada
tfm_utils.display(df_normalized)

,2015_porcentaje_de_poblacion_de_65_o_mas,2016_porcentaje_de_poblacion_de_65_o_mas,2017_porcentaje_de_poblacion_de_65_o_mas,2018_porcentaje_de_poblacion_de_65_o_mas,2019_porcentaje_de_poblacion_de_65_o_mas,2020_porcentaje_de_poblacion_de_65_o_mas,2021_porcentaje_de_poblacion_de_65_o_mas,2022_porcentaje_de_poblacion_de_65_o_mas,2023_porcentaje_de_poblacion_de_65_o_mas,nombre
0,25.3,25.4,25.6,25.9,25.8,26.0,26.5,27.1,27.5,abanilla
1,17.1,17.1,17.1,17.4,17.5,17.5,17.5,17.3,17.5,abaran
2,15.6,15.6,16.0,16.6,16.8,17.0,17.2,17.7,18.1,aguilas
3,19.3,20.5,20.5,19.3,19.7,19.3,19.1,19.4,20.4,albudeite
4,13.4,13.6,13.8,14.1,14.4,14.6,14.9,15.1,15.3,alcantarilla
5,24.1,24.1,24.2,24.6,25.3,25.7,24.8,24.5,25.2,aledo
6,12.9,13.6,13.7,13.9,14.0,14.1,14.0,14.2,14.4,alguazas
7,14.8,15.1,15.1,15.2,15.2,15.4,15.5,15.4,15.7,alhama_de_murcia
8,14.9,15.0,14.9,14.7,14.8,14.9,14.9,14.8,14.8,archena
9,12.3,12.5,12.7,13.1,13.1,13.2,13.4,13.6,13.9,beniel


In [6]:
# Ruta destino de la Delta Table
delta_path = tfm_utils.full_table_path(table_name)

print(f"Escribiendo {df_normalized.count()} registros en la Delta Table: {table_name}")
print(f"Destino: {delta_path}")

# Escritura en modo overwrite
(df_normalized
    .write
    .mode("overwrite")
    .format("delta")
    .save(delta_path)
)

print("¡Escritura en Delta Lake completada con éxito!")

Escribiendo 45 registros en la Delta Table: raw.crem.porcentaje_de_poblacion_de_65_o_mas
Destino: /tfm/delta_lake/raw/crem/porcentaje_de_poblacion_de_65_o_mas
¡Escritura en Delta Lake completada con éxito!
